In [ ]:
# From the project root, install the optional notebook dependencies with:
# python -m pip install -r notebooks/requirements.txt

## Project Overview

This notebook is a local semantic-retrieval experiment over PDF, TXT, and Markdown files in the repository's `docs/` directory. It uses Sentence Transformers to embed document chunks and queries, then stores and searches those vectors in a dedicated ChromaDB collection.

The notebook demonstrates document loading, chunking, embedding, vector storage, and top-k retrieval. It does not call an LLM to generate answers; the web app provides that Gemini-backed step using its own embedding model and separate ChromaDB collection.

In [ ]:
from pathlib import Path

import chromadb
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "docs").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DOCS_DIR = PROJECT_ROOT / "docs"
VECTOR_STORE_DIR = PROJECT_ROOT / "notebooks" / "vector_store"


def load_documents(folder=DOCS_DIR):
    if not folder.is_dir():
        raise FileNotFoundError(f"Document folder not found: {folder}")

    documents = []
    for path in sorted(folder.rglob("*")):
        if path.suffix.lower() == ".pdf":
            documents.extend(PyPDFLoader(str(path)).load())
        elif path.suffix.lower() in {".txt", ".md"}:
            documents.extend(TextLoader(str(path), encoding="utf-8").load())

    if not documents:
        raise ValueError(f"No PDF, TXT, or Markdown documents found in {folder}")

    return documents

### Document Loading and Preprocessing

The loader scans the repository's `docs/` directory recursively. PDF files are loaded page by page with `PyPDFLoader`; TXT and Markdown files are loaded with `TextLoader`. Source metadata is retained so retrieved passages can be traced to their files.

Add supported documents to the existing `docs/` directory before running the ingestion cells.

In [ ]:
"""Research-Oriented RAG Pipeline

This notebook demonstrates document ingestion, chunking, local embedding generation,
vector indexing, and semantic retrieval over this project's docs directory. It is a
retrieval-focused companion to the Gemini-backed web app; answer generation is handled
by the app, not this notebook.
"""

In [ ]:
all_documents = load_documents()
print(f"Loaded {len(all_documents)} documents/pages from {DOCS_DIR}")

### Chunks

In [26]:
# chunks
# !pip install langchain_text_splitters

In [ ]:
# Research-oriented chunking strategy

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""],
    )
    return text_splitter.split_documents(documents)

# Example usage:
# chunks = split_docs(all_documents, chunk_size=500, chunk_overlap=50)

In [ ]:
chunks = split_docs(all_documents)
print(f"Created {len(chunks)} chunks")

### Embedding

In [35]:
from sentence_transformers import SentenceTransformer

In [ ]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading embedding model:", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("Embedding dimension:", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, texts):
        embeddings = self.model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
        print("Embedding shape:", embeddings.shape)
        return embeddings


In [ ]:
embedding_manager = EmbeddingManager()

### Vector Store

In [40]:
import chromadb
import uuid

In [ ]:
import uuid


class VectorStoreManager:
    def __init__(self, persist_directory=VECTOR_STORE_DIR, collection_name="notebook_documents"):
        self.collection_name = collection_name
        self.persist_directory = Path(persist_directory)
        self.persist_directory.mkdir(parents=True, exist_ok=True)
        self.client = chromadb.PersistentClient(path=str(self.persist_directory))
        self.collection = self._get_or_create_collection()

    def _get_or_create_collection(self):
        return self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={
                "description": "Standalone notebook RAG retrieval collection",
                "hnsw:space": "cosine",
            },
        )

    def reset_collection(self):
        try:
            self.client.delete_collection(name=self.collection_name)
        except Exception:
            pass
        self.collection = self.client.create_collection(
            name=self.collection_name,
            metadata={
                "description": "Standalone notebook RAG retrieval collection",
                "hnsw:space": "cosine",
            },
        )

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents does not match number of embeddings")

        ids = [f"chunk_{i}_{uuid.uuid4().hex}" for i in range(len(documents))]
        metadatas = []
        for index, document in enumerate(documents):
            metadata = dict(document.metadata)
            metadata["chunk_index"] = index
            metadata["content_length"] = len(document.page_content)
            metadatas.append(metadata)

        self.collection.add(
            ids=ids,
            metadatas=metadatas,
            documents=[document.page_content for document in documents],
            embeddings=[embedding.tolist() for embedding in embeddings],
        )
        print(f"Stored {len(documents)} chunks in {self.persist_directory}")

In [ ]:
vector_store = VectorStoreManager()
texts = [document.page_content for document in chunks]
embeddings = embedding_manager.generate_embeddings(texts)

vector_store.reset_collection()
vector_store.add_documents(chunks, embeddings)

# Retrieval and Semantic Search

The retrieval phase is the core of a RAG system. Given a user query, the model converts the question into an embedding and searches the vector store for the most relevant document chunks.

### Retrieval Strategy
- Query embedding generation
- Semantic search in the vector database
- Ranking by similarity score
- Filtering using a threshold if needed

### Evaluation Focus
The retrieval stage can be evaluated using:
- precision@k
- recall@k
- mean reciprocal rank (MRR)
- answer relevance measured by human or model-based evaluation


In [ ]:
# ChromaDB cosine distance is converted to similarity in RAGRetriever.

In [ ]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        collection = self.vector_store.collection
        collection_count = collection.count()
        if collection_count == 0:
            return []

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        results = collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=min(top_k, collection_count),
            include=["documents", "metadatas", "distances", "ids"],
        )

        retrieved_docs = []
        for rank, (doc_id, metadata, document, distance) in enumerate(
            zip(
                results["ids"][0],
                results["metadatas"][0],
                results["documents"][0],
                results["distances"][0],
            ),
            start=1,
        ):
            similarity_score = 1 - distance
            if similarity_score >= score_threshold:
                retrieved_docs.append({
                    "id": doc_id,
                    "document": document,
                    "metadata": metadata,
                    "distance": distance,
                    "similarity_score": similarity_score,
                    "rank": rank,
                })

        print(f"Retrieved {len(retrieved_docs)} relevant chunks")
        return retrieved_docs

In [58]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [ ]:
query = "What is the company's leave policy?"
retrieved = rag_retriever.retrieve(query, top_k=5)

for item in retrieved:
    source = item["metadata"].get("source", "unknown")
    print(f"Rank: {item['rank']}, Similarity: {item['similarity_score']:.4f}, Source: {source}")
    print(item["document"][:400])
    print("-" * 80)